# Convenio de consignación vigente — MYM BOBINADOS, bodega 59

Diagnóstico del funcionamiento, uso, nivel pactado y retorno del convenio. El análisis
trabaja en modo lectura y cruza ventas, snapshots mensuales y el acuerdo exacto SKF.

La identidad principal es el `idproducto` exacto porque el acuerdo distingue variantes
con y sin grasa GJN. La llave canónica solo se utiliza como apoyo para detectar compras
equivalentes de otra marca.

## Parámetros y acuerdo pactado

Las reglas de decisión son visibles y ajustables. La cobertura objetivo conserva la
regla simple usada para SERMOTOR.

In [1]:
from io import StringIO
from pathlib import Path
import math
import sqlite3
import numpy as np
import pandas as pd
from IPython.display import display

from scripts.analyze_sermotor_consignacion import normalize_designation, ceil_even

IDBODEGA = 59
DB_PATH = Path("database/commercial.db")
MESES_COBERTURA_OBJETIVO = 2.5
COBERTURA_BAJA_MESES = 2.0
COBERTURA_ALTA_MESES = 8.0

ACUERDO_CSV = """codigo,marca,unidades_pactadas
400160SKF,SKF,3
400200SKF,SKF,10
400250SKF,SKF,9
400300SKF,SKF,6
400350SKF,SKF,8
400400SKF,SKF,4
400450SKF,SKF,4
400500SKF,SKF,2
400550SKF,SKF,2
400600SKF,SKF,1
400650SKF,SKF,5
400700SKF,SKF,4
400750SKF,SKF,5
400800SKF,SKF,3
400850SKF,SKF,5
400900SKF,SKF,4
401000SKF,SKF,1
6004-2Z/C3GJNSKF,SKF,10
6005-2RSH/C3GJNSKF,SKF,10
6005-2Z/C3GJNSKF,SKF,10
6006-2Z/C3GJNSKF,SKF,1
6007-2RS1/C3GJNSKF,SKF,9
6008-2Z/C3GJNSKF,SKF,2
6200-2Z/C3GJNSKF,SKF,4
6201-2RSH/C3GJNSKF,SKF,5
6201-2Z/C3GJNSKF,SKF,6
6202-2RSH/C3GJNSKF,SKF,18
6202-2Z/C3GJNSKF,SKF,7
6203-2RSH/C3GJNSKF,SKF,4
6203-2Z/C3GJNSKF,SKF,6
6204-2RSH/C3GJNSKF,SKF,7
6204-2Z/C3GJNSKF,SKF,10
6205-2RSH/C3GJNSKF,SKF,5
6205-2Z/C3GJNSKF,SKF,8
6206-2Z/C3GJNSKF,SKF,5
6207-2RS1/C3GJNSKF,SKF,9
6207-2Z/C3GJNSKF,SKF,5
6208-2RS1/C3GJNSKF,SKF,6
6208-2Z/C3GJNSKF,SKF,5
6209-2RS1/C3GJNSKF,SKF,3
6209-2Z/C3GJNSKF,SKF,6
6210-2RS1/C3GJNSKF,SKF,6
6210-2Z/C3GJNSKF,SKF,9
6211-2RS1/C3GJNSKF,SKF,5
6211-2RS1/C3SKF,SKF,1
6211-2Z/C3SKF,SKF,4
6212-2RS1/C3GJNSKF,SKF,5
6212-2Z/C3GJNSKF,SKF,4
6213-2RS1/C3GJNSKF,SKF,4
6213-2RS1/C3SKF,SKF,2
6213-2Z/C3GJNSKF,SKF,8
6213-2Z/C3SKF,SKF,1
6214-2RS1/C3SKF,SKF,4
6214-2Z/C3SKF,SKF,3
6215-2RS1/C3SKF,SKF,5
6215-2Z/C3GJNSKF,SKF,2
6216-2Z/C3SKF,SKF,2
6217-2Z/C3SKF,SKF,2
6218-2Z/C3SKF,SKF,1
6300-2RSH/C3GJNSKF,SKF,5
6300-2Z/C3SKF,SKF,4
6302-2RSH/C3GJNSKF,SKF,4
6302-2Z/C3GJNSKF,SKF,6
6303-2RSH/C3GJNSKF,SKF,2
6303-2Z/C3GJNSKF,SKF,2
6304-2RSH/C3GJNSKF,SKF,4
6304-2Z/C3GJNSKF,SKF,4
6305-2RS1/C3GJNSKF,SKF,3
6305-2Z/C3GJNSKF,SKF,2
6306-2RS1/C3GJNSKF,SKF,4
6306-2Z/C3GJNSKF,SKF,6
6307-2RS1/C3GJNSKF,SKF,8
6307-2Z/C3GJNSKF,SKF,4
6308-2RS1/C3GJNSKF,SKF,7
6308-2Z/C3GJNSKF,SKF,8
6309-2RS1/C3GJNSKF,SKF,11
6309-2Z/C3GJNSKF,SKF,5
6310-2RS1/C3GJNSKF,SKF,4
6310-2RS1/C3SKF,SKF,4
6310-2Z/C3GJNSKF,SKF,9
6311-2RS1/C3GJNSKF,SKF,3
6311-2Z/C3GJNSKF,SKF,5
6312-2RS1/C3GJNSKF,SKF,4
6312-2Z/C3GJNSKF,SKF,4
6313-2RS1/C3GJNSKF,SKF,5
6313-2Z/C3GJNSKF,SKF,2
6313-2Z/C3SKF,SKF,1
6314-2RS1/C3GJNSKF,SKF,4
6314-2Z/C3GJNSKF,SKF,2
6314-2Z/C3SKF,SKF,4
6315-2RS1/C3GJNSKF,SKF,3
6315-2Z/C3GJNSKF,SKF,3
6316-2RS1/C3SKF,SKF,3
6316-2Z/C3SKF,SKF,2
6316/C3SKF,SKF,2
6319/C3SKF,SKF,2
7312 BECBJSKF,SKF,2
99235SKF,SKF,2
NJ 220 ECJ/C3SKF,SKF,1
NU 2211 ECP/C3SKF,SKF,1"""
acuerdo = pd.read_csv(StringIO(ACUERDO_CSV), dtype={"codigo": str})
acuerdo["codigo"] = acuerdo["codigo"].str.strip()
assert not acuerdo["codigo"].duplicated().any()

pd.set_option("display.max_rows", 250)
pd.set_option("display.max_columns", 60)
PESO = "${:,.0f}".format

## Paso 0 — Identidad y calidad de datos

Se verifica el cliente dominante, los cruces exactos del acuerdo y las fechas efectivas
de snapshot. Un mes ausente no se interpola ni se inventa.

In [2]:
uri = f"file:{DB_PATH}?mode=ro"
with sqlite3.connect(uri, uri=True) as con:
    ventas = pd.read_sql_query(
        "SELECT * FROM raw_sales WHERE CAST(idbodega AS TEXT)=?",
        con, params=(str(IDBODEGA),),
    )
    inventario = pd.read_sql_query(
        "SELECT * FROM inventario_snapshot WHERE CAST(idbodega AS TEXT)=?",
        con, params=(str(IDBODEGA),),
    )

ventas["fecha"] = pd.to_datetime(ventas["fecha"], errors="raise")
inventario["fecha_snapshot"] = pd.to_datetime(
    inventario["fecha_snapshot"], errors="raise"
)
identidad = (
    ventas.groupby(["nit", "razonsocial"], dropna=False)
    .agg(lineas=("idproducto", "size"), fecha_min=("fecha", "min"),
         fecha_max=("fecha", "max"))
    .reset_index().sort_values("lineas", ascending=False)
)
display(identidad)

fechas_snapshot = sorted(inventario["fecha_snapshot"].dropna().unique())
calendario = pd.DataFrame({"fecha_snapshot": fechas_snapshot})
calendario["mes"] = calendario["fecha_snapshot"].dt.strftime("%Y-%m")
esperados = pd.date_range(
    calendario["fecha_snapshot"].min(),
    calendario["fecha_snapshot"].max(), freq="MS",
)
faltantes = sorted(set(esperados) - set(fechas_snapshot))
display(calendario)
print("Meses faltantes:", ", ".join(pd.Timestamp(x).strftime("%Y-%m") for x in faltantes) or "Ninguno")

sales_codes = set(ventas["idproducto"].dropna().astype(str))
inventory_codes = set(inventario["idproducto"].dropna().astype(str))
cruce = acuerdo.assign(
    existe_ventas=acuerdo["codigo"].isin(sales_codes),
    existe_inventario=acuerdo["codigo"].isin(inventory_codes),
)
no_cruzan = cruce[~cruce["existe_ventas"] | ~cruce["existe_inventario"]]
display(no_cruzan)
print(
    f"Acuerdo: {len(acuerdo)} referencias; "
    f"sin cruce completo: {len(no_cruzan)}."
)

,nit,razonsocial,lineas,fecha_min,fecha_max
1,805015031,MYM BOBINADOS INDUSTRIALES SAS,1203,2023-01-18,2025-10-16
0,805015031,MYM BOBINADOS INDUSTRIALES S.A.S.,250,2025-10-27,2026-07-14


,fecha_snapshot,mes
0,2026-01-01,2026-01
1,2026-02-01,2026-02
2,2026-03-01,2026-03
3,2026-04-01,2026-04
4,2026-05-01,2026-05
5,2026-06-01,2026-06
6,2026-07-01,2026-07


Meses faltantes: Ninguno


,codigo,marca,unidades_pactadas,existe_ventas,existe_inventario
20,6006-2Z/C3GJNSKF,SKF,1,True,False
44,6211-2RS1/C3SKF,SKF,1,True,False
45,6211-2Z/C3SKF,SKF,4,True,False
48,6213-2RS1/C3GJNSKF,SKF,4,True,False
51,6213-2Z/C3SKF,SKF,1,True,False
52,6214-2RS1/C3SKF,SKF,4,False,False
53,6214-2Z/C3SKF,SKF,3,True,False
54,6215-2RS1/C3SKF,SKF,5,True,False
56,6216-2Z/C3SKF,SKF,2,False,False
58,6218-2Z/C3SKF,SKF,1,True,False


Acuerdo: 100 referencias; sin cruce completo: 24.


## Bloque A — ¿El proceso funciona?

Sin movimientos de traslado no existe una medición independiente del consumo implícito:
la reposición es una incógnita. Para no fingir precisión, se muestran tres cantidades:

- **Consumo sin reposición:** stock inicial menos stock final.
- **Facturado:** ventas netas de la bodega durante el intervalo.
- **Reposición implícita:** facturado + stock final − stock inicial; es la reposición
  necesaria para reconciliar ambos datos.

Una reposición implícita negativa se marca como brecha porque apunta a transferencias,
ajustes, mermas, devoluciones o diferencias de corte.

Con reposición semanal, un inventario plano o cercano al pactado **no es evidencia de
sobrestock**. Las columnas de ceros e inventario medio frente al pacto se conservan
únicamente para diagnosticar si el proceso de reposición parece cumplirse. Los snapshots
mensuales no permiten comprobar el comportamiento semana a semana.

In [3]:
start = calendario["fecha_snapshot"].min()
end = calendario["fecha_snapshot"].max()
ventas_periodo = ventas[ventas["fecha"].ge(start) & ventas["fecha"].lt(end)].copy()

stock = inventario.pivot_table(
    index="idproducto", columns="fecha_snapshot", values="unidades",
    aggfunc="sum", fill_value=0,
)
intervalos = []
for left, right in zip(fechas_snapshot[:-1], fechas_snapshot[1:]):
    sold = (
        ventas[ventas["fecha"].ge(left) & ventas["fecha"].lt(right)]
        .groupby("idproducto")["cantidad"].sum()
    )
    keys = stock.index.union(sold.index)
    s0 = stock[left].reindex(keys, fill_value=0)
    s1 = stock[right].reindex(keys, fill_value=0)
    billed = sold.reindex(keys, fill_value=0)
    part = pd.DataFrame({
        "periodo": f"{pd.Timestamp(left):%Y-%m-%d} → {pd.Timestamp(right):%Y-%m-%d}",
        "idproducto": keys,
        "stock_inicial": s0.values,
        "stock_final": s1.values,
        "consumo_sin_reposicion": (s0 - s1).values,
        "facturado": billed.values,
        "reposicion_implicita": (billed + s1 - s0).values,
    })
    part["brecha_sin_reposicion"] = (
        part["consumo_sin_reposicion"] - part["facturado"]
    )
    intervalos.append(part)
movimientos = pd.concat(intervalos, ignore_index=True)
resumen_intervalos = (
    movimientos.groupby("periodo")
    .agg(
        stock_inicial=("stock_inicial", "sum"),
        stock_final=("stock_final", "sum"),
        consumo_sin_reposicion=("consumo_sin_reposicion", "sum"),
        facturado=("facturado", "sum"),
        reposicion_implicita=("reposicion_implicita", "sum"),
        brecha_sin_reposicion=("brecha_sin_reposicion", "sum"),
        referencias_reposicion_negativa=(
            "reposicion_implicita", lambda x: int((x < 0).sum())
        ),
    ).reset_index()
)
display(resumen_intervalos)

acuerdo_stock = stock.reindex(acuerdo["codigo"]).fillna(0)
ventas_por_codigo = ventas_periodo.groupby("idproducto")["cantidad"].sum()
zero_counts = (acuerdo_stock == 0).sum(axis=1)
flat = pd.DataFrame({
    "codigo": acuerdo["codigo"],
    "stock_inicial": acuerdo["codigo"].map(acuerdo_stock.iloc[:, 0]),
    "stock_final": acuerdo["codigo"].map(acuerdo_stock.iloc[:, -1]),
    "desviacion_snapshots": acuerdo["codigo"].map(acuerdo_stock.std(axis=1)),
    "consumo_facturado": acuerdo["codigo"].map(ventas_por_codigo).fillna(0),
    "veces_en_cero": acuerdo["codigo"].map(zero_counts).fillna(
        len(fechas_snapshot)
    ).astype(int),
    "inventario_promedio": acuerdo["codigo"].map(
        acuerdo_stock.mean(axis=1)
    ).fillna(0),
})
flat = flat.merge(
    acuerdo[["codigo", "unidades_pactadas"]], on="codigo", how="left"
)
flat["inventario_medio_vs_pactado"] = np.where(
    flat["unidades_pactadas"] > 0,
    flat["inventario_promedio"] / flat["unidades_pactadas"], np.nan,
)
flat["senal"] = np.select(
    [
        flat["consumo_facturado"].gt(0)
        & flat["inventario_promedio"].eq(0),
        flat["consumo_facturado"].gt(0)
        & flat["desviacion_snapshots"].fillna(0).eq(0),
    ],
    [
        "VENTA_SIN_CRUCE_INVENTARIO",
        "STOCK_PLANO_COMPATIBLE_REPOSICION_SEMANAL",
    ],
    default="SIN_ALERTA",
)
display(flat[flat["senal"] != "SIN_ALERTA"])

,periodo,stock_inicial,stock_final,consumo_sin_reposicion,facturado,reposicion_implicita,brecha_sin_reposicion,referencias_reposicion_negativa
0,2026-01-01 → 2026-02-01,391.0,363.0,28.0,114.0,86.0,-86.0,0
1,2026-02-01 → 2026-03-01,363.0,408.0,-45.0,0.0,45.0,-45.0,0
2,2026-03-01 → 2026-04-01,408.0,396.0,12.0,87.0,75.0,-75.0,0
3,2026-04-01 → 2026-05-01,396.0,377.0,19.0,23.0,4.0,-4.0,0
4,2026-05-01 → 2026-06-01,377.0,403.0,-26.0,39.0,65.0,-65.0,3
5,2026-06-01 → 2026-07-01,403.0,392.0,11.0,31.0,20.0,-20.0,1


,codigo,stock_inicial,stock_final,desviacion_snapshots,consumo_facturado,veces_en_cero,inventario_promedio,unidades_pactadas,inventario_medio_vs_pactado,senal
3,400300SKF,3.0,3.0,0.0,5.0,0,3.0,6,0.5,STOCK_PLANO_COMPATIBLE_REPOSICION_SEMANAL
82,6312-2RS1/C3GJNSKF,4.0,4.0,0.0,3.0,0,4.0,4,1.0,STOCK_PLANO_COMPATIBLE_REPOSICION_SEMANAL
92,6316-2RS1/C3SKF,0.0,0.0,0.0,1.0,7,0.0,3,0.0,VENTA_SIN_CRUCE_INVENTARIO
94,6316/C3SKF,0.0,0.0,0.0,8.0,7,0.0,2,0.0,VENTA_SIN_CRUCE_INVENTARIO


## Bloque B — ¿Se usa lo que está en la bodega?

La rotación se anualiza según los días realmente observados entre el primer y último
snapshot. El capital usa unidades físicas y costo promedio del snapshot más reciente.

In [4]:
days_observed = max((end - start).days, 1)
annual_factor = 365.25 / days_observed
latest_date = inventario["fecha_snapshot"].max()
latest = (
    inventario[inventario["fecha_snapshot"].eq(latest_date)]
    .sort_values("fecha_carga").drop_duplicates("idproducto", keep="last")
    .set_index("idproducto")
)
with sqlite3.connect(uri, uri=True) as con:
    costos_inventario = pd.read_sql_query(
        "SELECT idproducto, costo_unitario, fecha_snapshot, fecha_carga "
        "FROM inventario_snapshot WHERE costo_unitario IS NOT NULL", con,
    )
    costos_ventas = pd.read_sql_query(
        "SELECT idproducto, costo, fecha FROM raw_sales WHERE costo IS NOT NULL", con,
    )
costos_inventario["fecha_snapshot"] = pd.to_datetime(
    costos_inventario["fecha_snapshot"], errors="coerce"
)
costo_inv_actual = (
    costos_inventario.sort_values(["fecha_snapshot", "fecha_carga"])
    .drop_duplicates("idproducto", keep="last").set_index("idproducto")["costo_unitario"]
)
costos_ventas["fecha"] = pd.to_datetime(costos_ventas["fecha"], errors="coerce")
costo_venta_reciente = (
    costos_ventas.sort_values("fecha").drop_duplicates("idproducto", keep="last")
    .set_index("idproducto")["costo"]
)
avg_inventory = inventario.groupby("idproducto")["unidades"].mean()
first_inventory = (
    inventario[inventario["fecha_snapshot"].eq(start)]
    .groupby("idproducto")["unidades"].sum()
)
last_inventory = (
    inventario[inventario["fecha_snapshot"].eq(end)]
    .groupby("idproducto")["unidades"].sum()
)

uso = acuerdo[["codigo", "unidades_pactadas"]].copy()
uso["consumo_periodo"] = uso["codigo"].map(ventas_por_codigo).fillna(0)
uso["consumo_anualizado"] = uso["consumo_periodo"] * annual_factor
uso["inventario_promedio"] = uso["codigo"].map(avg_inventory).fillna(0)
uso["rotaciones_anualizadas"] = np.where(
    uso["inventario_promedio"] > 0,
    uso["consumo_anualizado"] / uso["inventario_promedio"], np.nan,
)
uso["inventario_inicial"] = uso["codigo"].map(first_inventory).fillna(0)
uso["inventario_final"] = uso["codigo"].map(last_inventory).fillna(0)
uso["inventario_sin_cambio"] = uso["inventario_inicial"].eq(uso["inventario_final"])
uso["costo_actual"] = uso["codigo"].map(costo_inv_actual)
uso["fuente_costo_actual"] = np.where(
    uso["costo_actual"].notna(), "ÚLTIMO_SNAPSHOT_EMPRESA", "ÚLTIMA_VENTA"
)
uso["costo_actual"] = uso["costo_actual"].fillna(
    uso["codigo"].map(costo_venta_reciente)
)
assert uso["costo_actual"].notna().all(), "Hay referencias pactadas sin costo actual"
uso["capital_comprometido_ref"] = (
    uso["unidades_pactadas"] * uso["costo_actual"]
)
display(uso.sort_values("rotaciones_anualizadas", na_position="first"))

capital_total = uso["capital_comprometido_ref"].sum()
capital_lento = uso.loc[
    uso["rotaciones_anualizadas"].fillna(0).lt(1), "capital_comprometido_ref"
].sum()
print(
    "Capital comprometido en referencias con rotación < 1x/año:",
    PESO(capital_lento),
    f"({capital_lento / capital_total:.1%} del capital del acuerdo)"
    if capital_total else "(sin capital)",
)

,codigo,unidades_pactadas,consumo_periodo,consumo_anualizado,inventario_promedio,rotaciones_anualizadas,inventario_inicial,inventario_final,inventario_sin_cambio,costo_actual,fuente_costo_actual,capital_comprometido_ref
20,6006-2Z/C3GJNSKF,1,0.0,0.000000,0.000000,NaN,0.0,0.0,True,18111.792858,ÚLTIMO_SNAPSHOT_EMPRESA,1.811179e+04
44,6211-2RS1/C3SKF,1,0.0,0.000000,0.000000,NaN,0.0,0.0,True,78150.389275,ÚLTIMO_SNAPSHOT_EMPRESA,7.815039e+04
45,6211-2Z/C3SKF,4,0.0,0.000000,0.000000,NaN,0.0,0.0,True,66096.557964,ÚLTIMO_SNAPSHOT_EMPRESA,2.643862e+05
48,6213-2RS1/C3GJNSKF,4,0.0,0.000000,0.000000,NaN,0.0,0.0,True,131629.311885,ÚLTIMO_SNAPSHOT_EMPRESA,5.265172e+05
51,6213-2Z/C3SKF,1,0.0,0.000000,0.000000,NaN,0.0,0.0,True,177091.103000,ÚLTIMA_VENTA,1.770911e+05
52,6214-2RS1/C3SKF,4,0.0,0.000000,0.000000,NaN,0.0,0.0,True,134649.659619,ÚLTIMO_SNAPSHOT_EMPRESA,5.385986e+05
53,6214-2Z/C3SKF,3,0.0,0.000000,0.000000,NaN,0.0,0.0,True,176032.240775,ÚLTIMO_SNAPSHOT_EMPRESA,5.280967e+05
54,6215-2RS1/C3SKF,5,0.0,0.000000,0.000000,NaN,0.0,0.0,True,223831.914905,ÚLTIMO_SNAPSHOT_EMPRESA,1.119160e+06
56,6216-2Z/C3SKF,2,0.0,0.000000,0.000000,NaN,0.0,0.0,True,203232.368893,ÚLTIMO_SNAPSHOT_EMPRESA,4.064647e+05
58,6218-2Z/C3SKF,1,0.0,0.000000,0.000000,NaN,0.0,0.0,True,467499.007682,ÚLTIMO_SNAPSHOT_EMPRESA,4.674990e+05


Capital comprometido en referencias con rotación < 1x/año: $19,997,324 (59.2% del capital del acuerdo)


## Bloque C — ¿Es correcta la cantidad pactada?

El nivel diagnóstico equivale a 2,5 meses de consumo anualizado, redondeado hacia arriba
a número par. El veredicto se decide exclusivamente por la cobertura del nivel pactado:
menos de 2 meses implica **SUBIR**, entre 2 y 8 meses **MANTENER**, y más de 8 meses
**BAJAR**. El consumo cero conserva el corte duro **RETIRAR**.

El inventario observado y los snapshots en cero no participan en esta decisión porque
la reposición semanal tiende a devolver el stock al nivel pactado.

In [5]:
monthly_factor = 12 * days_observed / 365.25
monthly_factor = max(monthly_factor, 1)
metricas = uso.copy()
metricas["consumo_mensual"] = metricas["consumo_periodo"] / monthly_factor
metricas["meses_cobertura"] = np.where(
    metricas["consumo_mensual"] > 0,
    metricas["unidades_pactadas"] / metricas["consumo_mensual"], np.nan,
)
metricas["nivel_simple"] = (
    metricas["consumo_mensual"] * MESES_COBERTURA_OBJETIVO
).map(ceil_even)
metricas["veces_en_cero"] = metricas["codigo"].map(zero_counts).fillna(
    len(fechas_snapshot)
).astype(int)
metricas["inventario_maximo"] = metricas["codigo"].map(
    acuerdo_stock.max(axis=1)
).fillna(0)
metricas["inventario_medio_vs_pactado"] = np.where(
    metricas["unidades_pactadas"] > 0,
    metricas["inventario_promedio"] / metricas["unidades_pactadas"], np.nan,
)

def verdict(row):
    if row["consumo_periodo"] <= 0:
        return "RETIRAR"
    if row["meses_cobertura"] < COBERTURA_BAJA_MESES:
        return "SUBIR"
    if row["meses_cobertura"] > COBERTURA_ALTA_MESES:
        return "BAJAR"
    return "MANTENER"

def reason(row):
    if row["veredicto"] == "RETIRAR":
        return (
            f"Sin consumo entre {start:%Y-%m-%d} y {end:%Y-%m-%d}; "
            "retiro por corte duro de consumo cero."
        )
    if row["veredicto"] == "SUBIR":
        return (
            f"Cobertura pactada {row['meses_cobertura']:.2f} meses, menor que "
            f"el umbral de {COBERTURA_BAJA_MESES:.0f} meses."
        )
    if row["veredicto"] == "BAJAR":
        return (
            f"Cobertura pactada {row['meses_cobertura']:.2f} meses, mayor que "
            f"el umbral de {COBERTURA_ALTA_MESES:.0f} meses."
        )
    return (
        f"Cobertura pactada {row['meses_cobertura']:.2f} meses, dentro del "
        f"rango de {COBERTURA_BAJA_MESES:.0f} a "
        f"{COBERTURA_ALTA_MESES:.0f} meses."
    )

metricas["veredicto"] = metricas.apply(verdict, axis=1)
metricas["motivo"] = metricas.apply(reason, axis=1)
metricas["unidades_propuestas"] = np.select(
    [
        metricas["veredicto"].eq("RETIRAR"),
        metricas["veredicto"].isin(["BAJAR", "SUBIR"]),
    ],
    [0, metricas["nivel_simple"]],
    default=metricas["unidades_pactadas"],
).astype(int)
metricas["capital_propuesto_ref"] = (
    metricas["unidades_propuestas"] * metricas["costo_actual"]
)
display(
    metricas.sort_values(
        ["veredicto", "capital_comprometido_ref"], ascending=[True, False]
    )
)

,codigo,unidades_pactadas,consumo_periodo,consumo_anualizado,inventario_promedio,rotaciones_anualizadas,inventario_inicial,inventario_final,inventario_sin_cambio,costo_actual,fuente_costo_actual,capital_comprometido_ref,consumo_mensual,meses_cobertura,nivel_simple,veces_en_cero,inventario_maximo,inventario_medio_vs_pactado,veredicto,motivo,unidades_propuestas,capital_propuesto_ref
92,6316-2RS1/C3SKF,3,1.0,2.017956,0.000000,NaN,0.0,0.0,True,496143.361597,ÚLTIMO_SNAPSHOT_EMPRESA,1.488430e+06,0.168163,17.839836,1,7,0.0,0.000000,BAJAR,"Cobertura pactada 17.84 meses, mayor que el um...",1,4.961434e+05
87,6314-2RS1/C3GJNSKF,4,1.0,2.017956,2.142857,0.941713,3.0,2.0,False,328238.276773,ÚLTIMO_SNAPSHOT_EMPRESA,1.312953e+06,0.168163,23.786448,1,0,3.0,0.535714,BAJAR,"Cobertura pactada 23.79 meses, mayor que el um...",1,3.282383e+05
90,6315-2RS1/C3GJNSKF,3,1.0,2.017956,1.000000,2.017956,0.0,1.0,False,415145.566421,ÚLTIMO_SNAPSHOT_EMPRESA,1.245437e+06,0.168163,17.839836,1,1,1.0,0.333333,BAJAR,"Cobertura pactada 17.84 meses, mayor que el um...",1,4.151456e+05
42,6210-2Z/C3GJNSKF,9,3.0,6.053867,6.000000,1.008978,4.0,6.0,False,50800.748290,ÚLTIMO_SNAPSHOT_EMPRESA,4.572067e+05,0.504489,17.839836,2,0,7.0,0.666667,BAJAR,"Cobertura pactada 17.84 meses, mayor que el um...",2,1.016015e+05
46,6212-2RS1/C3GJNSKF,5,1.0,2.017956,6.428571,0.313904,3.0,7.0,False,84098.509116,ÚLTIMO_SNAPSHOT_EMPRESA,4.204925e+05,0.168163,29.733060,1,0,7.0,1.285714,BAJAR,"Cobertura pactada 29.73 meses, mayor que el um...",1,8.409851e+04
80,6311-2RS1/C3GJNSKF,3,2.0,4.035912,3.428571,1.177141,3.0,5.0,False,136247.746680,ÚLTIMO_SNAPSHOT_EMPRESA,4.087432e+05,0.336326,8.919918,1,0,6.0,1.142857,BAJAR,"Cobertura pactada 8.92 meses, mayor que el umb...",1,1.362477e+05
43,6211-2RS1/C3GJNSKF,5,2.0,4.035912,4.285714,0.941713,5.0,3.0,False,75359.664812,ÚLTIMO_SNAPSHOT_EMPRESA,3.767983e+05,0.336326,14.866530,1,0,5.0,0.857143,BAJAR,"Cobertura pactada 14.87 meses, mayor que el um...",1,7.535966e+04
73,6308-2RS1/C3GJNSKF,7,4.0,8.071823,4.714286,1.712205,1.0,7.0,False,53198.037247,ÚLTIMO_SNAPSHOT_EMPRESA,3.723863e+05,0.672652,10.406571,2,0,7.0,0.673469,BAJAR,"Cobertura pactada 10.41 meses, mayor que el um...",2,1.063961e+05
74,6308-2Z/C3GJNSKF,8,5.0,10.089779,4.857143,2.077307,4.0,4.0,True,46474.057440,ÚLTIMO_SNAPSHOT_EMPRESA,3.717925e+05,0.840815,9.514579,4,0,7.0,0.607143,BAJAR,"Cobertura pactada 9.51 meses, mayor que el umb...",4,1.858962e+05
71,6307-2RS1/C3GJNSKF,8,3.0,6.053867,4.000000,1.513467,4.0,6.0,False,44212.908828,ÚLTIMO_SNAPSHOT_EMPRESA,3.537033e+05,0.504489,15.857632,2,0,6.0,0.500000,BAJAR,"Cobertura pactada 15.86 meses, mayor que el um...",2,8.842582e+04


## Bloque D — ¿Qué sobra y qué falta?

Se separan cuatro conversaciones: retiros, compras del cliente desde otras bodegas,
equivalentes de otra marca y referencias consumidas en la bodega 59 fuera del acuerdo.

In [6]:
print("Candidatas a retirar")
display(metricas[metricas["veredicto"].eq("RETIRAR")][
    ["codigo", "unidades_pactadas", "inventario_promedio",
     "capital_comprometido_ref", "motivo"]
])

dominant_nit = str(identidad.groupby("nit")["lineas"].sum().idxmax())
with sqlite3.connect(uri, uri=True) as con:
    ventas_cliente = pd.read_sql_query(
        "SELECT * FROM raw_sales WHERE CAST(nit AS TEXT)=?", con,
        params=(dominant_nit,),
    )
ventas_cliente["fecha"] = pd.to_datetime(ventas_cliente["fecha"], errors="raise")
cliente_periodo = ventas_cliente[
    ventas_cliente["fecha"].ge(start) & ventas_cliente["fecha"].lt(end)
].copy()
otras_bodegas = cliente_periodo[
    cliente_periodo["idbodega"].astype(str).ne(str(IDBODEGA))
]
compras_fuera = (
    otras_bodegas.groupby(["idproducto", "nombreproducto"])
    .agg(unidades=("cantidad", "sum"), documentos=("numero", "nunique"),
         bodegas=("idbodega", lambda x: ",".join(sorted(set(map(str, x))))))
    .reset_index().sort_values("unidades", ascending=False)
)
display(compras_fuera[compras_fuera["unidades"] > 0].head(100))

agreement_parsed = pd.DataFrame(
    acuerdo["codigo"].map(normalize_designation).tolist(), index=acuerdo.index
)
agreement_keys = set(agreement_parsed.loc[
    agreement_parsed["ok"].fillna(False), "llave_canonica"
])
sales_parsed = pd.DataFrame(
    cliente_periodo["prefijo_1"].map(normalize_designation).tolist(),
    index=cliente_periodo.index,
).add_prefix("parser_")
cliente_norm = pd.concat([cliente_periodo, sales_parsed], axis=1)
otras_marcas = cliente_norm[
    cliente_norm["parser_llave_canonica"].isin(agreement_keys)
    & cliente_norm["sufijo"].fillna("").ne("SKF")
]
substituciones = (
    otras_marcas.groupby(
        ["parser_llave_canonica", "sufijo", "idproducto"], dropna=False
    )["cantidad"].sum().reset_index(name="unidades")
    .sort_values("unidades", ascending=False)
)
display(substituciones[substituciones["unidades"] > 0])

consumo_59 = (
    ventas_periodo.groupby(["idproducto", "nombreproducto"])["cantidad"]
    .sum().reset_index(name="unidades")
)
fuera_acuerdo = consumo_59[
    ~consumo_59["idproducto"].isin(set(acuerdo["codigo"]))
    & consumo_59["unidades"].gt(0)
].sort_values("unidades", ascending=False)
display(fuera_acuerdo.head(100))

Candidatas a retirar


,codigo,unidades_pactadas,inventario_promedio,capital_comprometido_ref,motivo
1,400200SKF,10,4.000000,6.912321e+04,Sin consumo entre 2026-01-01 y 2026-07-01; ret...
2,400250SKF,9,2.000000,2.101160e+05,Sin consumo entre 2026-01-01 y 2026-07-01; ret...
12,400750SKF,5,2.000000,1.834009e+05,Sin consumo entre 2026-01-01 y 2026-07-01; ret...
17,6004-2Z/C3GJNSKF,10,1.000000,1.024990e+05,Sin consumo entre 2026-01-01 y 2026-07-01; ret...
19,6005-2Z/C3GJNSKF,10,10.000000,1.190999e+05,Sin consumo entre 2026-01-01 y 2026-07-01; ret...
20,6006-2Z/C3GJNSKF,1,0.000000,1.811179e+04,Sin consumo entre 2026-01-01 y 2026-07-01; ret...
21,6007-2RS1/C3GJNSKF,9,8.000000,1.951782e+05,Sin consumo entre 2026-01-01 y 2026-07-01; ret...
22,6008-2Z/C3GJNSKF,2,2.000000,6.172936e+04,Sin consumo entre 2026-01-01 y 2026-07-01; ret...
24,6201-2RSH/C3GJNSKF,5,3.000000,4.021134e+04,Sin consumo entre 2026-01-01 y 2026-07-01; ret...
27,6202-2Z/C3GJNSKF,7,11.000000,5.178006e+04,Sin consumo entre 2026-01-01 y 2026-07-01; ret...


,idproducto,nombreproducto,unidades,documentos,bodegas
40,MB 22FAG,ARANDELA DE FIJACION,4.0,2,50
37,E-5 ELEMENTREX,7300020-ELASTOMERO,4.0,2,50
34,BWW 120SEE,ARANDELA DE AJUSTE,3.0,2,50
12,55X70X8 VITONSOG,RETENEDOR DE ACEITE EN VITON,3.0,1,72
27,6322 M/C3VL0241SKF,RODAMIENTO RIGIDO DE BOLAS,2.0,1,50
47,RLS 11KOY,RODAMIENTO RIGIDO DE BOLAS,2.0,1,72
23,6319/C3SKF,RODAMIENTO RIGIDO DE BOLAS,2.0,1,50
46,QD-HINT,BUJE QD,2.0,1,72
48,RLS 12-2ZKOY,RODAMIENTO RIGIDO DE BOLAS,2.0,1,72
4,3213 A/C3SKF,RODAMIENTO DE CONTACTO ANGULAR,2.0,2,50


,parser_llave_canonica,sufijo,idproducto,unidades


,idproducto,nombreproducto,unidades
27,6206-2RS1/C3GJNSKF,RODAMIENTO RIGIDO DE BOLAS,15.0
23,6204-2RSH/C3SKF,RODAMIENTO RIGIDO DE BOLAS,4.0
37,6211-2Z/C3GJNSKF,RODAMIENTO RIGIDO DE BOLAS,4.0
65,6318/C3SKF,RODAMIENTO RIGIDO DE BOLAS,3.0
66,6319-2Z/C3SKF,RODAMIENTO RIGIDO DE BOLAS,3.0
13,400950SKF,V-RING 95mm - 3.3/4,2.0
15,401100SKF,V-RING 110mm - 4.1/2,2.0
63,6317-2RS1/C3SKF,RODAMIENTO RIGIDO DE BOLAS,2.0
64,6318-2Z/C3GJNSKF,RODAMIENTO RIGIDO DE BOLAS,2.0
59,6314/C3SKF,RODAMIENTO RIGIDO DE BOLAS,1.0


## Bloque E — Facturación, capital comprometido y escenario antes/después

La referencia financiera única del convenio es el **capital comprometido**:
`unidades pactadas × costo actual`. El costo actual se toma del snapshot más reciente
disponible en la empresa y, si la referencia no aparece en inventario, de su venta más
reciente.

El inventario físico promedio observado se informa aparte como métrica operativa; no se
usa como denominador de rotación o retorno. Julio contiene información hasta la última
fecha disponible en ventas y por ello puede ser un mes parcial.

In [7]:
ventas_fin = ventas[
    ventas["fecha"].ge(pd.Timestamp("2026-01-01"))
    & ventas["fecha"].lt(pd.Timestamp("2026-08-01"))
].copy()
ventas_fin["mes"] = ventas_fin["fecha"].dt.to_period("M")
ventas_fin["costo_vendido"] = ventas_fin["cantidad"] * ventas_fin["costo"]
mensual = (
    ventas_fin.groupby("mes")
    .agg(
        facturacion_bruta=("neto", "sum"),
        costo_vendido=("costo_vendido", "sum"),
    )
    .reindex(pd.period_range("2026-01", "2026-07", freq="M"), fill_value=0)
)
mensual["margen"] = mensual["facturacion_bruta"] - mensual["costo_vendido"]
mensual.index = mensual.index.astype(str)
mensual.index.name = "mes"
promedio = mensual.mean().to_frame().T
promedio.index = ["PROMEDIO_MENSUAL"]
mensual_con_promedio = pd.concat([mensual, promedio])
display(mensual_con_promedio.style.format({
    "facturacion_bruta": PESO, "costo_vendido": PESO, "margen": PESO,
}))
ultima_fecha_ventas = ventas_fin["fecha"].max()
print(f"Última fecha incluida en julio: {ultima_fecha_ventas:%Y-%m-%d}")

capital_comprometido = metricas["capital_comprometido_ref"].sum()
capital_propuesto = metricas["capital_propuesto_ref"].sum()
capital_liberado = capital_comprometido - capital_propuesto
reduccion_capital = (
    capital_liberado / capital_comprometido if capital_comprometido else np.nan
)
costo_anualizado = mensual["costo_vendido"].mean() * 12
margen_anualizado = mensual["margen"].mean() * 12

inventario["valor_fisico"] = inventario["unidades"] * inventario["costo_unitario"]
valor_fisico_por_snapshot = (
    inventario.groupby("fecha_snapshot")["valor_fisico"].sum().reset_index()
)
inventario_fisico_promedio = valor_fisico_por_snapshot["valor_fisico"].mean()

escenario = pd.DataFrame({
    "Escenario": ["ANTES — pacto vigente", "DESPUÉS — niveles propuestos"],
    "Capital comprometido": [capital_comprometido, capital_propuesto],
    "Rotación anualizada": [
        costo_anualizado / capital_comprometido,
        costo_anualizado / capital_propuesto,
    ],
    "Retorno anualizado": [
        margen_anualizado / capital_comprometido,
        margen_anualizado / capital_propuesto,
    ],
})
display(escenario.style.format({
    "Capital comprometido": PESO,
    "Rotación anualizada": "{:.2f}x",
    "Retorno anualizado": "{:.1%}",
}))

operacion = pd.DataFrame({
    "Métrica operativa (no capital contractual)": [
        "Inventario físico promedio observado",
        "Snapshots incluidos",
    ],
    "Valor": [PESO(inventario_fisico_promedio), str(len(fechas_snapshot))],
})
display(operacion)
display(valor_fisico_por_snapshot.style.format({"valor_fisico": PESO}))

print(
    f"Aplicar los veredictos libera {PESO(capital_liberado)} "
    f"({reduccion_capital:.1%}) de capital comprometido; con el mismo margen "
    f"mensual observado, el retorno anualizado pasa de "
    f"{margen_anualizado / capital_comprometido:.1%} a "
    f"{margen_anualizado / capital_propuesto:.1%}."
)

ventas["anio"] = ventas["fecha"].dt.year
crecimiento = (
    ventas.groupby("anio")
    .agg(unidades=("cantidad", "sum"), ventas_netas=("neto", "sum"),
         lineas=("idproducto", "size"))
    .reset_index()
)
display(crecimiento.style.format({"unidades": "{:,.0f}", "ventas_netas": PESO}))

,facturacion_bruta,costo_vendido,margen
2026-01,"$14,719,522","$8,568,572","$6,150,951"
2026-02,$0,$0,$0
2026-03,"$14,308,546","$8,061,376","$6,247,171"
2026-04,"$3,356,627","$1,846,364","$1,510,263"
2026-05,"$6,979,399","$5,289,015","$1,690,384"
2026-06,"$2,405,239","$1,417,228","$988,011"
2026-07,"$2,472,104","$1,510,538","$961,565"
PROMEDIO_MENSUAL,"$6,320,205","$3,813,299","$2,506,906"


Última fecha incluida en julio: 2026-07-14


,Escenario,Capital comprometido,Rotación anualizada,Retorno anualizado
0,ANTES — pacto vigente,"$33,780,419",1.35x,89.1%
1,DESPUÉS — niveles propuestos,"$14,157,043",3.23x,212.5%


,Métrica operativa (no capital contractual),Valor
0,Inventario físico promedio observado,"$20,258,283"
1,Snapshots incluidos,7


,fecha_snapshot,valor_fisico
0,2026-01-01 00:00:00,"$18,867,408"
1,2026-02-01 00:00:00,"$19,449,897"
2,2026-03-01 00:00:00,"$21,098,493"
3,2026-04-01 00:00:00,"$21,197,620"
4,2026-05-01 00:00:00,"$19,422,686"
5,2026-06-01 00:00:00,"$21,131,282"
6,2026-07-01 00:00:00,"$20,640,598"


Aplicar los veredictos libera $19,623,376 (58.1%) de capital comprometido; con el mismo margen mensual observado, el retorno anualizado pasa de 89.1% a 212.5%.


,anio,unidades,ventas_netas,lineas
0,2023,"1,008","$181,173,084",480
1,2024,826,"$114,948,576",451
2,2025,514,"$77,600,532",336
3,2026,311,"$44,241,437",186


## Tabla consolidada final

Cada fila corresponde a una referencia exacta del acuerdo. El capital comprometido usa
las unidades pactadas; el capital propuesto aplica el veredicto y el nivel simple. La
rotación por referencia y cobertura usan el periodo comparable entre snapshots.

In [8]:
final = metricas[[
    "codigo", "unidades_pactadas", "consumo_periodo", "consumo_mensual",
    "meses_cobertura", "veces_en_cero", "inventario_medio_vs_pactado",
    "rotaciones_anualizadas",
    "inventario_promedio", "inventario_final", "costo_actual",
    "capital_comprometido_ref", "unidades_propuestas",
    "capital_propuesto_ref", "veredicto", "motivo",
]].rename(columns={
    "codigo": "Referencia",
    "unidades_pactadas": "Pactado",
    "consumo_periodo": "Consumo 2026 comparable",
    "consumo_mensual": "Consumo mensual",
    "meses_cobertura": "Meses cobertura",
    "veces_en_cero": "Veces en cero",
    "inventario_medio_vs_pactado": "Inventario medio / pactado",
    "rotaciones_anualizadas": "Rotación anualizada",
    "inventario_promedio": "Inventario promedio",
    "inventario_final": "Inventario último snapshot",
    "costo_actual": "Costo actual",
    "capital_comprometido_ref": "Capital comprometido",
    "unidades_propuestas": "Unidades propuestas",
    "capital_propuesto_ref": "Capital propuesto",
    "veredicto": "Veredicto",
    "motivo": "Motivo",
})
orden = {"SUBIR": 0, "RETIRAR": 1, "BAJAR": 2, "MANTENER": 3}
final["_orden"] = final["Veredicto"].map(orden)
final = final.sort_values(
    ["_orden", "Capital comprometido"], ascending=[True, False]
).drop(columns="_orden")
display(final.style.format({
    "Pactado": "{:,.0f}",
    "Consumo 2026 comparable": "{:,.1f}",
    "Consumo mensual": "{:,.1f}",
    "Meses cobertura": "{:,.2f}",
    "Veces en cero": "{:,.0f}",
    "Inventario medio / pactado": "{:.0%}",
    "Rotación anualizada": "{:,.1f}",
    "Inventario promedio": "{:,.1f}",
    "Inventario último snapshot": "{:,.0f}",
    "Costo actual": PESO,
    "Capital comprometido": PESO,
    "Unidades propuestas": "{:,.0f}",
    "Capital propuesto": PESO,
}, na_rep="—"))

resumen_final = (
    final.groupby("Veredicto")
    .agg(referencias=("Referencia", "size"), pactado=("Pactado", "sum"),
         propuesto=("Unidades propuestas", "sum"),
         capital_antes=("Capital comprometido", "sum"),
         capital_despues=("Capital propuesto", "sum"))
    .reindex(["SUBIR", "MANTENER", "BAJAR", "RETIRAR"])
    .dropna(how="all").reset_index()
)
display(resumen_final.style.format({
    "referencias": "{:,.0f}", "pactado": "{:,.0f}", "propuesto": "{:,.0f}",
    "capital_antes": PESO, "capital_despues": PESO,
}))

,Referencia,Pactado,Consumo 2026 comparable,Consumo mensual,Meses cobertura,Veces en cero,Inventario medio / pactado,Rotación anualizada,Inventario promedio,Inventario último snapshot,Costo actual,Capital comprometido,Unidades propuestas,Capital propuesto,Veredicto,Motivo
94,6316/C3SKF,2,8.0,1.3,1.49,7,0%,—,0.0,0,"$506,666","$1,013,332",4,"$2,026,664",SUBIR,"Cobertura pactada 1.49 meses, menor que el umbral de 2 meses."
8,400550SKF,2,6.0,1.0,1.98,0,136%,4.5,2.7,1,"$37,163","$74,326",4,"$148,651",SUBIR,"Cobertura pactada 1.98 meses, menor que el umbral de 2 meses."
9,400600SKF,1,7.0,1.2,0.85,0,186%,7.6,1.9,1,"$12,324","$12,324",4,"$49,295",SUBIR,"Cobertura pactada 0.85 meses, menor que el umbral de 2 meses."
89,6314-2Z/C3SKF,4,0.0,0.0,—,7,0%,—,0.0,0,"$481,576","$1,926,304",0,$0,RETIRAR,Sin consumo entre 2026-01-01 y 2026-07-01; retiro por corte duro de consumo cero.
54,6215-2RS1/C3SKF,5,0.0,0.0,—,7,0%,—,0.0,0,"$223,832","$1,119,160",0,$0,RETIRAR,Sin consumo entre 2026-01-01 y 2026-07-01; retiro por corte duro de consumo cero.
91,6315-2Z/C3GJNSKF,3,0.0,0.0,—,0,100%,0.0,3.0,3,"$369,882","$1,109,646",0,$0,RETIRAR,Sin consumo entre 2026-01-01 y 2026-07-01; retiro por corte duro de consumo cero.
95,6319/C3SKF,2,0.0,0.0,—,7,0%,—,0.0,0,"$524,358","$1,048,716",0,$0,RETIRAR,Sin consumo entre 2026-01-01 y 2026-07-01; retiro por corte duro de consumo cero.
50,6213-2Z/C3GJNSKF,8,0.0,0.0,—,0,50%,0.0,4.0,4,"$126,997","$1,015,980",0,$0,RETIRAR,Sin consumo entre 2026-01-01 y 2026-07-01; retiro por corte duro de consumo cero.
93,6316-2Z/C3SKF,2,0.0,0.0,—,7,0%,—,0.0,0,"$320,505","$641,010",0,$0,RETIRAR,Sin consumo entre 2026-01-01 y 2026-07-01; retiro por corte duro de consumo cero.
52,6214-2RS1/C3SKF,4,0.0,0.0,—,7,0%,—,0.0,0,"$134,650","$538,599",0,$0,RETIRAR,Sin consumo entre 2026-01-01 y 2026-07-01; retiro por corte duro de consumo cero.


,Veredicto,referencias,pactado,propuesto,capital_antes,capital_despues
0,SUBIR,3,5,12,"$1,099,981","$2,224,610"
1,MANTENER,30,141,141,"$9,439,651","$9,439,651"
2,BAJAR,24,155,40,"$8,768,783","$2,492,781"
3,RETIRAR,43,165,0,"$14,472,004",$0


## Exportación opcional

El notebook es el entregable. Active la bandera solo si necesita extraer las tablas.

In [9]:
EXPORTAR = False
if EXPORTAR:
    output = Path("outputs/mym_bodega59")
    output.mkdir(parents=True, exist_ok=True)
    final.to_csv(output / "tabla_consolidada.csv", index=False)
    resumen_intervalos.to_csv(output / "reconciliacion_intervalos.csv", index=False)
    fuera_acuerdo.to_csv(output / "fuera_acuerdo.csv", index=False)
    print(f"Exportado en {output.resolve()}")
else:
    print("Exportación desactivada.")

Exportación desactivada.
